# Финансовая модель Тіл Mix
Валюта — KZT. Горизонт — три года. Расчёт ведётся в постоянных ценах без дисконтирования. Выгода — оценка высвобожденного времени, а не выручка. Открытый и управляемый варианты моделируют одну и ту же функциональность. Источники и границы модели описаны в [ЛЗ 3](lab-03.md).

In [1]:
from math import isclose

# Сценарный расчёт. Меняйте параметры здесь и запустите все ячейки заново.
years = 3
launch_rate = 5_500
content_rate = 3_000
qa_rate = 4_500
support_rate = 4_500
teacher_rate = 5_500

groups = 16
teaching_months = 8
session_hours_saved_per_group_month = 1.5
prep_hours_saved_per_group_year = 4
adoption = (0.50, 0.80, 1.00)

saas_price_usd_reference = 1_190
usd_to_kzt_assumption = 520
provider_monthly_budget = 10_000
saas_effect_factor = 0.80

steady_effect = groups * (
    teaching_months * session_hours_saved_per_group_month + prep_hours_saved_per_group_year
) * teacher_rate
annual_effect = [steady_effect * factor for factor in adoption]

## Затраты
Разработка оценена на 130 часов. Контент — 80 часов при запуске и 36 часов в год. QA и координация — 45 часов при запуске. Обучение — 16 часов при запуске и 8 часов в год. Сопровождение открытого варианта — 72 часа в год. Для SaaS учтены первичная настройка и перенос сценариев, ежегодная поддержка, контент, обучение и лицензия. В управляемом варианте дополнительный бюджет инфраструктуры не меняет предполагаемую экономию времени.

In [2]:
open_launch = {
    "Разработка": 130 * launch_rate,
    "Контент": 80 * content_rate,
    "QA и PM": 45 * qa_rate,
    "Обучение": 16 * content_rate,
}
open_year = {
    "Инфраструктура": 0,
    "Лицензии": 0,
    "Поддержка": 72 * support_rate,
    "Обновление контента": 36 * content_rate,
    "Обучение": 8 * content_rate,
}
managed_year = {
    **open_year,
    "Инфраструктура": 12 * provider_monthly_budget,
}
saas_launch = {
    "Настройка и перенос": 60 * launch_rate,
    "Контент": 80 * content_rate,
    "QA и PM": 35 * qa_rate,
    "Обучение": 16 * content_rate,
}
saas_year = {
    "Инфраструктура (в лицензии)": 0,
    "Лицензия": saas_price_usd_reference * usd_to_kzt_assumption,
    "Поддержка": 42 * support_rate,
    "Обновление контента": 36 * content_rate,
    "Обучение": 8 * content_rate,
}

## Формулы
TCO = стартовые затраты + сумма годовых расходов. ROI = (суммарная выгода − TCO) / TCO. Окупаемость — первый год, когда накопленная чистая выгода покрывает вложения. Внутри года месяцы интерполируются при равномерном поступлении эффекта и расходовании операционного бюджета. При нулевой инкрементальной стоимости ROI не рассчитывается.

In [3]:
def evaluate(launch, yearly, benefits, cost_multiplier=1.0, benefit_multiplier=1.0):
    initial = sum(launch.values()) * cost_multiplier
    recurring = sum(yearly.values()) * cost_multiplier
    tco = initial + recurring * len(benefits)
    gross = sum(benefits) * benefit_multiplier
    roi = (gross - tco) / tco if tco else None
    accumulated = -initial
    payback = None
    cash_flows = [-initial]
    for year, benefit in enumerate(benefits):
        net = benefit * benefit_multiplier - recurring
        if accumulated < 0 <= accumulated + net and payback is None:
            payback = (year - accumulated / net) * 12
        accumulated += net
        cash_flows.append(net)
    return {
        "initial": initial, "annual": recurring, "tco": tco, "effect": gross,
        "roi": roi, "payback": payback, "cash_flows": cash_flows,
    }


def kzt(value):
    return f"{value:,.0f}".replace(",", " ") + " ₸"


def print_table(results):
    for name, item in results.items():
        roi = "н/д" if item["roi"] is None else f'{item["roi"]:.1%}'
        payback = "н/д" if not item["tco"] else (
            "> 36 мес." if item["payback"] is None else f'{item["payback"]:.1f} мес.'
        )
        print(f'{name}: TCO {kzt(item["tco"])} | эффект {kzt(item["effect"])} | ROI {roi} | окупаемость {payback}')


assert len(adoption) == years
base = evaluate(open_launch, open_year, annual_effect)
solutions = {
    "Текущий процесс": evaluate({}, {}, [0] * years),
    "MoodleCloud": evaluate(saas_launch, saas_year, [saas_effect_factor * amount for amount in annual_effect]),
    "Тіл Mix открытый стек": base,
    "Тіл Mix управляемый": evaluate(open_launch, managed_year, annual_effect),
}
print_table(solutions)
print("Чистый ряд открытого варианта (год 0, год 1, год 2, год 3):")
print(" | ".join(kzt(amount) for amount in base["cash_flows"]))
assert isclose(sum(base["cash_flows"]), base["effect"] - base["tco"])

Текущий процесс: TCO 0 ₸ | эффект 0 ₸ | ROI н/д | окупаемость н/д
MoodleCloud: TCO 3 594 900 ₸ | эффект 2 590 720 ₸ | ROI -27.9% | окупаемость > 36 мес.
Тіл Mix открытый стек: TCO 2 573 500 ₸ | эффект 3 238 400 ₸ | ROI 25.8% | окупаемость 27.6 мес.
Тіл Mix управляемый: TCO 2 933 500 ₸ | эффект 3 238 400 ₸ | ROI 10.4% | окупаемость 31.6 мес.
Чистый ряд открытого варианта (год 0, год 1, год 2, год 3):
-1 205 500 ₸ | 248 000 ₸ | 670 400 ₸ | 952 000 ₸


## Чувствительность открытого варианта
Отдельно увеличиваем стоимость на 30%, уменьшаем эффект вдвое и применяем оба отклонения вместе. Денежные границы для решения о пилоте подтверждаются после замера потраченного преподавателем времени.

In [4]:
sensitivity = {
    "База": base,
    "Затраты +30%": evaluate(open_launch, open_year, annual_effect, cost_multiplier=1.3),
    "Эффект 50%": evaluate(open_launch, open_year, annual_effect, benefit_multiplier=0.5),
    "Оба отклонения": evaluate(open_launch, open_year, annual_effect, 1.3, 0.5),
}
print_table(sensitivity)

База: TCO 2 573 500 ₸ | эффект 3 238 400 ₸ | ROI 25.8% | окупаемость 27.6 мес.
Затраты +30%: TCO 3 345 550 ₸ | эффект 3 238 400 ₸ | ROI -3.2% | окупаемость > 36 мес.
Эффект 50%: TCO 2 573 500 ₸ | эффект 1 619 200 ₸ | ROI -37.1% | окупаемость > 36 мес.
Оба отклонения: TCO 3 345 550 ₸ | эффект 1 619 200 ₸ | ROI -51.6% | окупаемость > 36 мес.
